# 01 — Dataset Overview
### Mục 0.1 của đề cương — Khảo sát cấu trúc bộ dữ liệu CWRU

**Mục đích**: quét thư mục dữ liệu, dựng bảng manifest (nhãn, tải, đường
kính lỗi, vị trí cảm biến...), và chạy các sanity check quan trọng đã xác
định ở phần Tổng quan đề tài:

1. Sampling rate của file Normal baseline (nhiều nguồn không thống nhất
   12kHz hay 48kHz).
2. Đường kính lỗi 0.028"/0.040" dùng vòng bi **NTN**, khác SKF 6205 dùng
   cho 0.007"/0.014"/0.021".
3. Outer Race có 3 vị trí lỗi (6h/3h/12h) — đề tài chỉ dùng vị trí
   **Centered (6h)**.
4. RPM thực tế trong file có khớp RPM danh định theo tải hay không.

**Trước khi chạy với dữ liệu thật**: mở `common/io_utils.py`, chỉnh hàm
`parse_metadata_from_filename()` cho khớp cấu trúc thư mục/tên file bạn
đang có — file `.mat` gốc CWRU chỉ có tên số, không tự chứa metadata.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from common import io_utils, pipeline, synthetic
pd.set_option("display.max_colwidth", 120)

In [2]:
# ============================== CẤU HÌNH ==============================
# Đổi USE_SYNTHETIC_DATA = False và chỉnh REAL_DATA_ROOT khi đã có dữ liệu
# CWRU thật. Xem README.md phần "Chuyển sang dữ liệu thật".
USE_SYNTHETIC_DATA = True
REAL_DATA_ROOT = Path("../../data/raw")            # <-- de_tai_nckh/data/raw/
SYNTHETIC_DATA_ROOT = Path("./_data/synthetic_cwru")
OUTPUT_DIR = Path("./outputs")
FORCE_REBUILD_MANIFEST = False
# ========================================================================

In [3]:
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

> ⚠️ **Lưu ý dữ liệu giả lập**: khi `USE_SYNTHETIC_DATA = True`, mọi tín hiệu
> trong notebook này là **giả lập** (nhiễu + xung điều biên mô phỏng), chỉ để
> kiểm tra code chạy đúng và xem trước hình dạng đầu ra. **Không dùng số liệu
> giả lập này làm kết quả báo cáo chính thức.** Khi có dữ liệu CWRU thật, đổi
> `USE_SYNTHETIC_DATA = False` ở cell CẤU HÌNH và chạy lại toàn bộ notebook.

In [4]:
manifest = pipeline.get_manifest(
    use_synthetic=USE_SYNTHETIC_DATA,
    real_data_root=REAL_DATA_ROOT,
    synthetic_data_root=SYNTHETIC_DATA_ROOT,
    output_dir=OUTPUT_DIR,
    force_rebuild=FORCE_REBUILD_MANIFEST,
)
print(f"Tổng số file trong manifest: {len(manifest)}")
manifest.head()

Tổng số file trong manifest: 40


,file_path,load_hp,label,fault_diameter_mils,or_position,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,_data/synthetic_cwru/0hp/B_007.mat,0,B,7.0,NaN,120000,None,None,1797.0,None,,False
1,_data/synthetic_cwru/0hp/B_014.mat,0,B,14.0,NaN,120000,None,None,1797.0,None,,False
2,_data/synthetic_cwru/0hp/B_021.mat,0,B,21.0,NaN,120000,None,None,1797.0,None,,False
3,_data/synthetic_cwru/0hp/IR_007.mat,0,IR,7.0,NaN,120000,None,None,1797.0,None,,False
4,_data/synthetic_cwru/0hp/IR_014.mat,0,IR,14.0,NaN,120000,None,None,1797.0,None,,False


## Thống kê số file theo (nhãn × tải)

In [5]:
pivot = manifest.pivot_table(
    index="label", columns="load_hp", values="file_path",
    aggfunc="count", fill_value=0,
)
print(pivot)
pivot.to_csv(TABLES_DIR / "01_label_load_pivot.csv")

load_hp  0  1  2  3
label              
B        3  3  3  3
IR       3  3  3  3
Normal   1  1  1  1
OR       3  3  3  3


## Cảnh báo sanity check

Mỗi dòng cảnh báo cần được đọc và quyết định xử lý (giữ/loại/sửa) — script chỉ phát hiện, không tự động sửa.

In [6]:
n_warn = int(manifest["has_warning"].sum())
print(f"Số file có cảnh báo: {n_warn} / {len(manifest)}\n")

if n_warn > 0:
    warn_cols = ["file_path", "label", "load_hp", "fault_diameter_mils", "warnings"]
    warn_df = manifest.loc[manifest["has_warning"], warn_cols]
    warn_df.to_csv(TABLES_DIR / "01_warnings.csv", index=False)
    display(warn_df)
else:
    print("Không có cảnh báo nào trên bộ dữ liệu hiện tại.")

Số file có cảnh báo: 0 / 40

Không có cảnh báo nào trên bộ dữ liệu hiện tại.


## Tự kiểm chứng sanity check (bonus — không thuộc bể dữ liệu chính)

Phần dưới đây tạo riêng 4 file `.mat` **cố ý gài lỗi** (tách biệt hoàn
toàn khỏi `manifest` ở trên) để xác nhận cả 4 loại sanity check thực sự
hoạt động, trước khi bạn tin tưởng dùng chúng trên dữ liệu thật.

In [7]:
edge_root, expected_keywords = synthetic.build_edge_case_dataset(Path("./_data/edge_cases"))
edge_manifest = io_utils.build_manifest(edge_root)

all_warnings_text = " ".join(edge_manifest["warnings"].tolist())
print("Kiểm tra từng loại cảnh báo có xuất hiện không:\n")
all_ok = True
for kw in expected_keywords:
    found = kw in all_warnings_text
    print(f"  [{'✓ CÓ' if found else '✗ THIẾU'}] {kw}")
    all_ok = all_ok and found

print("\n✅ Toàn bộ sanity check hoạt động đúng." if all_ok else
      "\n❌ Có sanity check chưa hoạt động như mong đợi — kiểm tra lại common/io_utils.py.")

Kiểm tra từng loại cảnh báo có xuất hiện không:

  [✓ CÓ] NGHI_NGO_SAMPLING_RATE
  [✓ CÓ] OR_NGOAI_PHAM_VI
  [✓ CÓ] VONG_BI_NTN
  [✓ CÓ] RPM_LECH

✅ Toàn bộ sanity check hoạt động đúng.


## Checklist trước khi sang notebook 02

- [ ] Đã chỉnh `parse_metadata_from_filename()` khớp với dữ liệu thật.
- [ ] Đã đọc hết `outputs/tables/01_warnings.csv`, quyết định rõ giữ/loại
      từng trường hợp.
- [ ] Đã đối chiếu ngẫu nhiên vài dòng manifest với bảng tra cứu gốc trên
      trang CWRU Bearing Data Center để xác nhận parse đúng.